# EDF Control Vane Design  
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

This design uses **four (4) deflectable flat-plate vanes** placed in the **EDF exhaust stream**.

- The vanes **re-direct the jet flow** to generate control moments about all three axes:
  - Pitch  
  - Yaw  
  - Roll  

> These are **jet-pipe control vanes**, similar in concept to those used in **rocket nozzles**

---

## Mixing Law

**Reference frame:**  
### Aft View — Looking Upstream Into the Jet

|        |        |        |
|--------|--------|--------|
|        | **T (+z)** |        |
|        |    │    |        |
| **L (+y)** |    **+**    | **R (-y)** |
|        |    │    |        |
|        | **B (-z)** |        |


### Control Contribution Table

| Vane | Pitch | Yaw | Roll |
|------|------|------|------|
| **T** (Top)    | +d_p | 0    | +d_r |
| **B** (Bottom) | +d_p | 0    | -d_r |
| **L** (Left)   | 0    | +d_y | -d_r |
| **R** (Right)  | 0    | +d_y | +d_r |

---

## Control Interpretation

- **Pitch-up**  
  → Top (T) and Bottom (B) vanes deflect **together**

- **Yaw-right**  
  → Left (L) and Right (R) vanes deflect **together**

- **Roll-right**  
  → All four vanes deflect **differentially**

---

## Inputs

- `out/sizing_result.yaml` *(from Notebook 1)*  
- `out/airfoil.yaml` *(from Notebook 2)*  

---

## Outputs

- `out/control_vanes.yaml`  
- `out/vane_authority_curves.png`  
- `out/vane_geometry.png`  

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point
from conceptual_design.control_vane_design import (
    VaneParams, design_vanes, mixing_check, write_control_vanes_yaml,
)
from conceptual_design import reports
from conceptual_design.plots import plot_vane_authority, plot_vane_geometry

nb = nb_setup()
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

*** Nothing is hard-coded here. All numbers come from first and second notebooks. 

In [ ]:
# -- Re-run the sizing loop from config/ -- same design point as NB2 -----------
inp, result = solve_design_point(CONFIG_PATH)

MTOW_kg      = result.m_total_kg
W_N          = MTOW_kg * inp.env.g
chord_mean_m = result.wing.chord_mean

# wing airfoil from NB2 - for section comparison only, NOT for vane design
af = load_handoff(OUT_PATH, "airfoil")
wing_af_name = af["designation"]     # NACA 2412

print(f"Converged MTOW : {MTOW_kg:.3f} kg  ({W_N:.2f} N)")
print(f"D_rotor        : {inp.rotor.D_rotor_m*1e3:.1f} mm")
print(f"Wing chord     : {chord_mean_m*1e3:.1f} mm")
print(f"Wing airfoil   : {wing_af_name}  (NB2 reference only)")

# EDF Jet Parameters

---

## Actuator-Disk (Momentum) Theory

$$
v_h = \sqrt{\frac{T}{2 \rho A_{disk}}}
$$
*Induced hover velocity*

$$
V_{jet} = 2 v_h
$$
*Velocity at vane plane*

$$
q_{jet} = \frac{1}{2} \rho V_{jet}^2
$$
*Dynamic pressure on vanes*

---

## Interpretation

- $q_{jet}$ drives all vane forces  
- It is typically **several times larger** than the freestream cruise dynamic pressure  

In [ ]:
# -- Vane design parameters (all design choices in one place) ------------------
p = VaneParams(
    tc_vane           = 0.02,    # flat-plate thickness ratio
    Re_correction     = 0.85,    # Pelletier & Mueller 2000, Re < 60k
    delta_stall_deg   = 15.0,    # soft flat-plate stall onset
    delta_max_deg     = 20.0,    # mechanical deflection limit
    delta_design_deg  = 10.0,    # authority-check deflection
    kappa_hub         = 0.40,    # hub radius / tip radius
    AR_vane           = 2.5,     # vane aspect ratio h/c
    n_vanes           = 4,       # T / B / L / R
    span_frac         = 0.85,    # vane span / annulus height
    hinge_xc          = 0.25,    # hinge line chord fraction
    C_hm_coeff        = 0.5,     # hinge-moment coefficient factor
    SF_servo          = 2.5,     # servo torque safety factor
    servo_margin      = 1.2,     # extra servo torque margin
    body_len_factor   = 1.2,     # slender-body length / wing MAC
    cg_frac           = 0.55,    # CG station / body length
    fus_radius_factor = 1.05,    # bulk fuselage radius / R_tip
)

d = design_vanes(
    MTOW_kg=MTOW_kg, D_rotor_m=inp.rotor.D_rotor_m,
    V_cruise=inp.mission.V_cruise, rho=inp.env.rho, g=inp.env.g,
    chord_mean_m=chord_mean_m, ddot_min_deg_s2=inp.aero.ddot_min_deg_s2, p=p,
)

print(f"R_tip   : {d.R_tip*1e3:.1f} mm")
print(f"v_h     : {d.v_h:.3f} m/s")
print(f"V_jet   : {d.V_jet:.3f} m/s")
print(f"q_jet   : {d.q_jet:.2f} Pa  ({d.q_jet/d.q_fs:.1f}x cruise q)")

# Section 3 — Section Selection

---

## Why Not the Wing Airfoil?

The vanes and the wing operate in fundamentally different regimes:

### Comparison

| Parameter        | Wing (NB2)            | Control Vane              |
|------------------|----------------------|---------------------------|
| **Chord Re**     | 100,000 – 300,000    | 15,000 – 60,000           |
| **Operating α**  | 2–5° (fixed)         | 0–20° (full sweep)        |
| **Thickness role** | Structural spar     | Thin → less blockage      |
| **Key metric**   | L/D in cruise        | Linear \( C_l(\delta) \) over 0–20° |
| **Stall behavior** | Gentle onset       | Must be **monotonic**     |

---

### Key Insight

At $$ Re < 60000 $$, thick cambered sections (e.g. NACA 2412) develop a **laminar separation bubble** near the leading edge.

- This causes a **lift coefficient plateau or reversal**
- Occurs **well below 10°**
- This behavior is **dangerous for control surfaces**

---

## Candidate Comparison

| Section                          | Use? | Reason |
|----------------------------------|------|--------|
| **NACA 2412 (NB2 wing)**         | ❌ NO | Laminar bubble at \( Re < 60k \). Non-monotonic \( C_l(\delta) \) below 10° |
| **NACA 65-006**                  | ❌ NO | Compressor stator section. Same low-Re bubble issue |
| **Double Circular Arc (t/c = 4%)** | ⚠️ OK | No sharp LE. Only marginal improvement over flat plate |
| **Flat Plate (t/c = 2%) ← BEST** | ✅ YES | Most linear \( C_l(\delta) \) at low Re. No LE radius → no bubble. Monotonic to 15°+. Minimum blockage. Easily laser-cut CFRP |

---

## Conclusion

The **flat plate (t/c ≈ 2%)** is the optimal choice for EDF control vanes because:

- Ensures **monotonic control authority**
- Avoids **low-Re laminar separation issues**
- Minimizes **flow blockage**
- Enables **simple, low-cost manufacturing**

In [ ]:
print(f"Flat plate t/c = {p.tc_vane:.2f}  (wing section {wing_af_name} rejected at vane Re)")
print(f"Cl_alpha_eff   = 2*pi x {p.Re_correction:.2f} = {d.Cl_alpha_eff:.3f} /rad")
print(f"Re_vane        = {d.Re_vane:.0f}  (low-Re flat-plate regime)")

# Section 4 — Vane Geometry Sizing

---

## Design Requirement

Size the vanes to meet a minimum angular acceleration:

$$
\ddot{\theta}_{\min} = 30^\circ / \text{s}^2
$$

---

## Vane Force Model

Force generated by a single vane at design deflection \( \delta_d \):

$$
F_{\text{vane}} = q_{\text{jet}} \; S_{\text{vane}} \; C_{l,\alpha}^{\text{eff}} \; \delta_d
$$

$$
S_{\text{vane}} = c_{\text{vane}} \; h_{\text{vane}}
$$

---

## Vane Geometry

The vane spans the **hub-to-tip annulus**:

$$
h_{\text{vane}} = \text{span}_{\text{frac}} \; R_{\text{tip}} \; (1 - \kappa)
$$

$$
c_{\text{vane}} = \frac{h_{\text{vane}}}{AR_{\text{vane}}}
$$

---

## Moment Arms

- **Pitch / Yaw**
  $$
  \rightarrow L_{\text{CG}}
  $$
  *(Axial distance: EDF exit plane to aircraft center of gravity)*

- **Roll**
  $$
  \rightarrow R_{\text{mid}}
  $$
  *(Mid-span radius of vane)*

---

## Moments of Inertia  
*(Slender cylinder approximation)*

$$
I_{\text{pitch}} = I_{\text{yaw}} = \frac{m}{12} \left(3 R_{\text{fus}}^2 + L_{\text{body}}^2 \right)
$$

$$
I_{\text{roll}} = \frac{m}{2} R_{\text{fus}}^2
$$

---

## Interpretation

- Required moment:
  $$
  M = I \cdot \ddot{\theta}_{\min}
  $$

- This sets the **minimum required vane force**, and therefore:
  - vane area $ S_{\text{vane}} $
  - vane span $ h_{\text{vane}} $
  - vane chord $ c_{\text{vane}} $

---


In [ ]:
print(f"R_tip / R_hub : {d.R_tip*1e3:.1f} / {d.R_hub*1e3:.1f} mm  (kappa = {p.kappa_hub:.2f})")
print(f"h_vane        : {d.h_vane*1e3:.2f} mm  (span_frac {p.span_frac:.2f})")
print(f"c_vane        : {d.c_vane*1e3:.2f} mm  (AR = {p.AR_vane:.1f})")
print(f"S_vane        : {d.S_vane*1e4:.3f} cm^2 per vane")
print(f"Moment arms   : L_CG = {d.L_CG*1e3:.1f} mm, L_roll = {d.L_roll*1e3:.1f} mm")

In [ ]:
print(f"Authority at delta = {p.delta_design_deg:.0f} deg  "
      f"(requirement {d.ddot_min:.0f} deg/s^2):")
for axis, M, a in [("pitch", d.M_pitch_Nm, d.ddot_pitch),
                   ("yaw",   d.M_yaw_Nm,   d.ddot_yaw),
                   ("roll",  d.M_roll_Nm,  d.ddot_roll)]:
    print(f"  {axis:<6}: M = {M*1e3:8.3f} N*mm   ddot = {a:7.1f} deg/s^2  "
          f"{'OK' if a >= d.ddot_min else 'FAIL'}")

# Deflection Limits and Thrust Loss

---

## Blockage Estimate (First-Order)

Blocked area per vane:

$$
A_{\text{block}} = S_{\text{vane}} \; \left| \sin(\delta) \right|
$$

---

## Thrust Loss Estimate

Percentage thrust loss due to vane blockage:

$$
\text{thrust}_{\text{loss},\%} =
\left(
\frac{n_{\text{vanes}} \; A_{\text{block}}}{A_{\text{disk}}}
\right) \times 100
$$

---

## Interpretation

- Thrust loss increases with:
  - vane deflection angle $ \delta $
  - vane area $ S_{\text{vane}} $
  - number of vanes $ n_{\text{vanes}} $

- For small angles:
  $$
  \sin(\delta) \approx \delta
  $$
  → thrust loss scales approximately **linearly with deflection**

- This creates a key design trade-off:
  - **More control authority** ↔ **Higher thrust loss**

---

In [ ]:
plot_vane_authority(d, OUT_PATH / "vane_authority_curves.png")

# Section 7 — Hinge Moment and Actuator Sizing

---

## Hinge Moment Model  
*(Flat plate, hinge at quarter-chord)*

$$
C_{hm} = 0.5 \; C_{l,\alpha}^{eff} \; \delta
$$
*Hinge moment coefficient (restoring, magnitude)*

$$
M_{hinge} = q_{jet} \; S_{vane} \; c_{vane} \; C_{hm}
$$

---

## Servo Torque Requirement

Including safety factor and additional margin:

$$
M_{servo} = M_{hinge,\max} \times SF \times 1.2
$$

---

## Unit Conversion

Servo specifications are typically given in **g·cm**:

$$
1 \; N·m = 10\,197 \; g·cm
$$

$$
M_{servo}^{(g\dot{c} cm)} = M_{servo}^{(N\dot{c} m)} \times 10\,197
$$

---

## Interpretation

- Hinge moment scales with:
  - jet dynamic pressure $ q_{jet} $
  - vane area $ S_{vane} $
  - vane chord $ c_{vane} $
  - deflection angle $ \delta $

- Design implications:
  - High $ q_{jet} $ → **very high torque demand**
  - Small geometry changes → **large actuator impact**
  - Safety factor $ SF $ is critical for reliability

---

In [ ]:
print(f"M_hinge @ delta_max   : {d.M_hinge_max*1e3:.4f} N*mm")
print(f"Required servo torque : >= {d.servo_torque_req_gcm:.1f} g*cm  "
      f"(SF={p.SF_servo}, +{(p.servo_margin-1)*100:.0f}% margin)")

# Section 8 — Control Mixing Matrix

---

## Definition

- **Rows (actuators):** T, B, L, R  
- **Columns (commands):** Pitch, Yaw, Roll  

Each element represents the **gain applied to a normalized command** in the range \([-1, +1]\).

---

## Mixing Matrix

$$
\begin{bmatrix}
T \\
B \\
L \\
R
\end{bmatrix}
=
\begin{bmatrix}
+1 &  0 & +1 \\
+1 &  0 & -1 \\
 0 & +1 & -1 \\
 0 & +1 & +1
\end{bmatrix}
\begin{bmatrix}
\delta_{pitch} \\
\delta_{yaw} \\
\delta_{roll}
\end{bmatrix}
$$

---

## Interpretation

- **Pitch control**
  - T and B move **together**

- **Yaw control**
  - L and R move **together**

- **Roll control**
  - All four vanes move **differentially**

---

## Notes

- Commands $ \delta_{pitch}, \delta_{yaw}, \delta_{roll} \in [-1, +1] $
- Matrix can be scaled by:
  - maximum deflection $ \delta_{max} $
  - or actuator limits

---

In [ ]:
reports.print_mixing_table(mixing_check(d), p.delta_max_deg)

# Section 9 — Geometry Visualisation

---

## Overview

This section defines the required plots for visualizing the vane geometry and deflection behavior.

---

## Plot 1 — Duct Cross-Section (Aft View)

- View: **Looking upstream into the EDF jet**
- Geometry:
  - Circular duct (radius \( R_{tip} \))
  - Hub region (optional, radius \( R_{hub} \))
- Vanes:
  - Four vanes (**T, B, L, R**) shown as **radial lines**
  - Positioned at:
    - T → \( +z \)
    - B → \( -z \)
    - L → \( +y \)
    - R → \( -y \)

### Purpose

- Verify **spatial arrangement**
- Confirm **symmetry**
- Validate **annulus coverage**

---

## Plot 2 — Vane Profile (Flat Plate)

- Section: **2D profile view**
- Geometry:
  - Flat plate with chord $ c_{vane} $
  - Thickness $ t/c \approx 2\% $

- Deflection angles:

$$
\delta \in \{0^\circ, 5^\circ, 10^\circ, 15^\circ, 20^\circ\}
$$

### Purpose

- Visualize **deflection envelope**
- Check for:
  - potential **flow blockage**
  - possible **mechanical interference**
- Illustrate **linear control region**

---

## Notes

- Plot 1 is best implemented using:
  - `matplotlib` (top view)
  - or CAD export for documentation

- Plot 2:
  - Overlay multiple deflections in a single figure
  - Use consistent pivot point (hinge at quarter-chord)

---

In [ ]:
plot_vane_geometry(d, OUT_PATH / "vane_geometry.png")

# Section 10 — Output Export

---

## Purpose

Export vane design data for downstream use in:

- CAD geometry generation  
- Control-law development notebooks  

---

## Output File

$$
\texttt{out/control\_vanes.yaml}
$$

---

## Contents

The YAML file should include:

- **Geometry**
  - $ c_{vane} $ (chord)
  - $ h_{vane} $ (span)
  - aspect ratio $ AR_{vane} $

- **Aerodynamics**
  - $ C_{l,\alpha}^{eff} $
  - deflection limits $ \delta_{min}, \delta_{max} $

- **Jet parameters**
  - $ q_{jet} $
  - $ V_{jet} $

- **Control mapping**
  - mixing matrix
  - scaling factors

---

## Role in Pipeline

This file acts as a **handoff interface** between:

- **Sizing notebook → Geometry / CAD**
- **Sizing notebook → Control-law notebook**

---

- Prefer **SI units throughout**

---

In [ ]:
write_control_vanes_yaml(d, OUT_PATH / "control_vanes.yaml")
print(f"Vane design written -> {OUT_PATH / 'control_vanes.yaml'}")

# Section 11 — Design Summary

---

In [ ]:
reports.print_vane_summary(d, wing_af_name)